# LELA60331 Week 12 Seminar: Sequence models

Our classifiers so far have represented a text as a one-hot vector: which words occur in it, but **not in what order**. Yet order matters in language: *the dog bit the man* and *the man bit the dog* contain the same words. This week we use **recurrent neural networks** (RNNs), which read a sequence one item at a time, to classify:

1. surnames by nationality, reading them **one character at a time**
2. reviews and utterances, reading them **one word at a time**

We use PyTorch, as in Week 11. If you are using Colab, switch on a GPU (*Runtime → Change runtime type → GPU*): the later parts are slow without one.

| Part | Topic |
|---|---|
| 1 | Recurrent neural networks |
| 2 | Surnames as sequences of characters |
| 3 | An RNN surname classifier |
| 4 | Words as vectors: embeddings and hierarchical indexing |
| 5 | Classifying reviews and utterances |

**Lectures:** this sheet builds on the Week 9 lecture (word embeddings), the Week 11 lecture on recurrent neural networks (in particular classification with RNNs), and this week's lecture (LSTMs).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import gen_batches
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using", device)

---
# Part 1 · Recurrent neural networks

An RNN reads a sequence **one step at a time**. At each step it combines the current input (say, one character) with its **hidden state**, a vector that summarises what it has read so far, to produce a new hidden state. After the last step, the final hidden state summarises the whole sequence, and we pass it to an ordinary linear layer to classify it.

```
   input:    B   a   n   n   a   r   d
             ↓   ↓   ↓   ↓   ↓   ↓   ↓
   hidden:  h1→ h2→ h3→ h4→ h5→ h6→ h7 ──→ linear layer ──→ class scores
```

The same weights are used at every step, so an RNN can read sequences of any length.

In PyTorch, `nn.RNN(input_size, hidden_size, batch_first=True)` is an RNN layer. Its input is a 3-dimensional tensor with the shape **[number of sequences, number of steps, size of each input]** (`batch_first=True` means the sequences come first). It returns the hidden state at **every** step, with the shape [sequences, steps, hidden size], plus the final hidden state separately.

In [ ]:
torch.manual_seed(0)
example = torch.randn(1, 10, 6)          # 1 sequence, 10 steps, 6 numbers per step
rnn = nn.RNN(input_size=6, hidden_size=4, batch_first=True)
all_steps, final = rnn(example)
print(all_steps.shape)                   # the hidden state at each of the 10 steps
print(all_steps[:, -1, :])               # the hidden state at the last step...
print(final)                             # ...is the same as the final hidden state

---
# Part 2 · Surnames as sequences of characters

Our first data set has just over 10,000 surnames, each labelled with one of 18 nationalities.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/surnames_data.csv

In [ ]:
surnames_df = pd.read_csv("surnames_data.csv")
print(surnames_df.head())
print(surnames_df["nationality"].value_counts())    # how many names of each nationality

To feed a name to an RNN we turn each character into a **one-hot vector** over all the characters that occur in the data. So a name becomes a matrix with one row per character.

The RNN layer needs all the sequences in a batch to have the **same length**, so shorter names are **padded** with rows of zeros. We put the padding at the **start**, so that every name ends at the last step, where we will read off the hidden state.

```
"Abe" (max length 6):   [0 0 0 ...]   ← padding
                         [0 0 0 ...]   ← padding
                         [0 0 0 ...]   ← padding
                         [1 0 0 ...]   ← A
                         [0 1 0 ...]   ← b
                         [0 0 1 ...]   ← e
```

### 🏋️ Exercise 1 — Characters to tensors

1. `chars`: a sorted list of all the different characters in the surnames. `char_to_index` then gives each character's position. *(done for you)*
2. Complete the loop that fills `X`, a tensor of shape [number of names, longest name length, number of characters]. For name `i`, character `k` goes in row `start + k`, where `start` is the number of padding rows, and in the column for that character.
3. `y`: a tensor of each name's nationality **number** (its position in `nationalities`).

In [ ]:
names = list(surnames_df["surname"])
nationalities = sorted(set(surnames_df["nationality"]))
chars = sorted(set("".join(names)))
char_to_index = {}
for j in range(len(chars)):
    char_to_index[chars[j]] = j
max_len = max([len(name) for name in names])
print(len(names), "names,", len(chars), "characters, longest name:", max_len)

X = torch.zeros(len(names), max_len, len(chars))
for i in range(len(names)):
    name = names[i]
    start = ???                     # the number of padding rows
    for k in range(len(name)):
        X[i, ???, ???] = 1

y = ???
print(X.shape, y.shape)

In [ ]:
# Check: the last 3 rows of "Abboud" (name 1) should be one-hot vectors for o, u and d
print(names[1], [chars[j] for j in X[1].argmax(dim=1)[-3:]], X[1].sum(dim=1))

---
# Part 3 · An RNN surname classifier

Here is the model. It is a class, as in Week 11. `forward` runs the RNN over the names, takes the hidden state at the **last** step (`x[:, -1, :]`), and passes it to a linear layer that gives a score for each of the 18 nationalities.

We use `nn.CrossEntropyLoss`, PyTorch's loss for multiclass classification. It applies softmax itself, so the model returns plain scores, and it takes the correct class **numbers** as targets. We also use the **Adam** optimiser, which adjusts the learning rate for each weight as it goes and usually trains faster than plain gradient descent.

In [ ]:
class SeqModel(nn.Module):
    def __init__(self, input_size, hidden_size, n_classes):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        x, _ = self.rnn(x)              # the hidden states at every step
        x = x[:, -1, :]                 # take only the last step
        return self.linear(x)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)
X_train, X_test, y_train, y_test = X_train.to(device), X_test.to(device), y_train.to(device), y_test.to(device)

### 🏋️ Exercise 2 — Training

Complete the training step inside the loop (Week 11, Exercise 3): reset the gradients, run the model on the batch, calculate the loss, work out the gradients and update the weights. Training takes about a minute without a GPU.

In [ ]:
def train_seq(model, X_train, y_train, loss_fn, n_epochs=50, batch_size=128, lr=0.005):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_loss = []
    for epoch in range(n_epochs):
        model.train()
        total = 0.0
        for s in gen_batches(len(X_train), batch_size):
            inputs, targets = X_train[s], y_train[s]
            ???
            pred = ???
            loss = ???
            ???
            ???
            total += loss.item()
        epoch_loss.append(total)
    plt.plot(epoch_loss)
    plt.xlabel("number of epochs")
    plt.ylabel("loss")
    plt.show()

torch.manual_seed(0)
rnn_model = SeqModel(len(chars), 42, len(nationalities))
train_seq(rnn_model, X_train, y_train, nn.CrossEntropyLoss())

### 🏋️ Exercise 3 — Evaluation

1. Calculate `y_test_pred`, the predicted nationality number for each test name (the position of the highest score: `.argmax(dim=1)`). The line `with torch.no_grad():` switches off gradient recording, as in Week 11.
2. The scikit-learn function `precision_recall_fscore_support(true, predicted, average="macro")` returns the macro-averaged precision, recall and F-score (and a fourth value we can ignore). It needs ordinary arrays, so `.cpu()` moves the tensors back from the GPU first. Print the accuracy and the macro-averaged scores.

Why might the macro-averaged scores be much lower than the accuracy? *Hint:* look at the number of names of each nationality.

In [ ]:
def evaluate(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        y_test_pred = ???
    accuracy = (y_test_pred == y_test).float().mean().item()
    p, r, f, _ = ???
    print("accuracy:", round(accuracy, 3), " macro precision:", round(p, 3), " recall:", round(r, 3))

evaluate(rnn_model, X_test, y_test)

### 🏋️ Exercise 4 — Try your own name

Complete `predict_nationality(model, name)`, which turns a name into a padded one-hot tensor of shape [1, max_len, number of characters] (as in Exercise 1, for one name) and returns the predicted nationality. Try it on your own surname and some others. (Any character that doesn't occur in the data is skipped.)

In [ ]:
def predict_nationality(model, name):
    one_hot = torch.zeros(1, max_len, len(chars))
    start = max_len - len(name)
    for k in range(len(name)):
        if name[k] in char_to_index:
            one_hot[0, ???, ???] = 1
    model.eval()
    with torch.no_grad():
        scores = ???
    return ???

for name in ["Bannard", "Nakamura", "Rossi", "Schmidt", "Kowalski"]:
    print(name, predict_nationality(rnn_model, name))

### LSTMs

Simple RNNs find it hard to remember information from many steps back, because it gets overwritten at each step. A **long short-term memory** network (**LSTM**), covered in this week's lecture, is an RNN with **gates**: learned switches that decide, at each step, what to keep in its memory, what to forget and what to output. This lets it carry information over longer sequences. In PyTorch, `nn.LSTM` is used just like `nn.RNN`.

### 🏋️ Exercise 5 — An LSTM surname classifier

Complete `LSTMModel` by replacing the RNN layer with an `nn.LSTM` layer, then train and evaluate it. Does it do better than the RNN?

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, n_classes):
        super().__init__()
        self.lstm = ???
        self.linear = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        x, _ = ???
        x = x[:, -1, :]
        return self.linear(x)

torch.manual_seed(0)
lstm_model = LSTMModel(len(chars), 42, len(nationalities))
train_seq(lstm_model, X_train, y_train, nn.CrossEntropyLoss())
evaluate(lstm_model, X_test, y_test)

---
# Part 4 · Words as vectors: embeddings and hierarchical indexing

To read a text **word by word**, we need a vector for each word. A one-hot vector over a large vocabulary would be very long, and would treat every pair of words as equally different. Instead we use **word embeddings**: each word is represented by a vector of, say, 300 numbers, learned from a large corpus so that words used in similar contexts (such as *good* and *great*) have similar vectors (the distributional idea from LELA60341 Week 8). The Week 9 lecture explains how embeddings such as word2vec are learned. The data in Part 5 has already been converted to embeddings for you.

### Hierarchical indexing

When each text is a sequence of words, it is natural to label each row of a table with **two** things: which text it belongs to, and its position in that text. pandas allows this with a **hierarchical index** (a *MultiIndex*), where each row has a pair of labels. Here is a small example: 3 "texts" with 2 to 4 "words" each.

In [ ]:
s = pd.Series([0.5, 0.1, 0.9, 0.3, 0.7, 0.2, 0.8, 0.4, 0.6],
              index=[[0, 0, 0, 1, 1, 2, 2, 2, 2],                 # which text
                     ["the", "cat", "sat", "a", "dog", "my", "big", "red", "car"]])   # which word
print(s)
print(s.loc[2])               # everything for text 2
print(s.loc[0, "cat"])        # text 0, word "cat"

### 🏋️ Exercise 6 — Selecting from a hierarchical index

1. `text_1`: everything for text 1
2. `n_words`: a list of the number of words in each of the three texts. *Hint:* `len(s.loc[i])`.

In [ ]:
text_1 = ???
n_words = ???
print(text_1)
print(n_words)          # [3, 2, 4]

---
# Part 5 · Classifying reviews and utterances

## Yelp reviews

Next we apply the same approach to the Yelp sentiment data from earlier weeks. To keep training time manageable, we use a random sample of 1000 reviews. Each word is represented by a 300-number embedding, and every review is padded (at the start) to the length of the longest, 887 words. So the data is a 1000 × 887 × 300 tensor. The labels are 1 for positive and 0 for negative.

The next cell downloads the two files from Google Drive and unzips them. Loading the reviews takes a moment: the file is about 1 GB.

In [ ]:
!gdown 19cYQ_B3diu6RqlpYT5n9qHvS08_cScp7
!gdown 1DHj5zFiWX3hF3o8RxMx2Hn4sOW-VsHTs
!gunzip -kf reviews_for_rnn.pt.gz
reviews_emb = torch.load("reviews_for_rnn.pt")
review_labels = torch.load("review_labels_for_rnn.pt").float()
print(reviews_emb.shape, review_labels.shape)

R_train, R_test, r_train, r_test = train_test_split(reviews_emb, review_labels, test_size=0.2, random_state=30)
R_train, R_test, r_train, r_test = R_train.to(device), R_test.to(device), r_train.to(device), r_test.to(device)

### 🏋️ Exercise 7 — An LSTM sentiment classifier

This is a **binary** task, so:

* the model needs a **single** output, passed through `torch.sigmoid`; `.squeeze(1)` then turns the column of outputs into a plain vector (as in Week 11)
* the loss is `nn.BCELoss()`
* a review is predicted positive if its output is above 0.5

Complete the model, then train it. Each epoch takes a while without a GPU, so we only train for a few epochs. Then calculate the precision and recall on the test set.

In [ ]:
class ReviewLSTM(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = ???
        self.linear = ???

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        return ???

torch.manual_seed(0)
review_model = ReviewLSTM()
train_seq(review_model, R_train, r_train, nn.BCELoss(), n_epochs=8, batch_size=32, lr=0.002)

review_model.eval()
with torch.no_grad():
    r_pred = ???
tp = ???
fp = ???
fn = ???
print("precision:", tp / max(tp + fp, 1), " recall:", tp / max(tp + fn, 1))

With only 800 training reviews and a few epochs, don't expect this to beat the logistic regression of earlier weeks, which was trained on 8000 reviews. Try training for more epochs if you have a GPU.

## Intent classification

Finally, the intent classification data from Week 10. Each of the 2100 utterances has been converted into word embeddings and padded (at the start) to 23 words, giving a 2100 × 23 × 300 tensor. The intents are given as one-hot vectors over the seven intents.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/utts_emb.pt.gz
!gunzip -f utts_emb.pt.gz
!wget https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/intents_emb.pt
X_utts = torch.load("utts_emb.pt")
y_intents = torch.load("intents_emb.pt")
print(X_utts.shape, y_intents.shape)

### 🏋️ Exercise 8 — An LSTM intent classifier

1. `intent_numbers`: the intent number of each utterance (the position of the 1 in its one-hot vector)
2. Split the data 80/20 into training and test sets (as above, with `random_state=30`), and move them to `device`.
3. Create an `LSTMModel` (from Exercise 5) with input size 300, hidden size 64 and 7 classes. Train it with `train_seq` and `nn.CrossEntropyLoss()` for 30 epochs, and evaluate it with `evaluate`. How does it compare with the softmax classifier from Week 10?

In [ ]:
# Exercise 8

### ⭐ Exercise 9 (challenge) — Preparing the data yourself

In Exercise 8 the padded tensor was prepared for you. Here you build it yourself, from a pandas DataFrame with a **hierarchical index**: each row is the 300-number embedding of one word, and the index gives (utterance number, word position). Some words had no embedding and were left out, so utterances have different numbers of rows.

Build a tensor `X_mine` of shape [2100, 23, 300], with each utterance's embeddings placed at the **end** of its 23 rows (padding at the start). *Hint:* `torch.tensor(intent_df.loc[i].values)` gives the embeddings of utterance `i` as a tensor. Check your result against `X_utts` with `torch.allclose(X_mine, X_utts)`.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60342/refs/heads/main/intent_classification.pickle
intent_df = pd.read_pickle("intent_classification.pickle")
print(intent_df.shape)
print(intent_df.iloc[:5, :4])

In [ ]:
# Exercise 9

---
## ✏️ Summary

This week you have:

* used **recurrent neural networks**, which read a sequence one step at a time and summarise it in a hidden state
* turned names into padded sequences of one-hot character vectors, and trained RNN and **LSTM** classifiers on them
* met **word embeddings** and pandas **hierarchical indexing**
* built LSTM classifiers for reviews and utterances that read them word by word